In [1]:
import json
import pandas as pd
import pickle

In [2]:
data_processed = pd.read_csv("../data/02-processed-properati.csv", sep=',', index_col=0)
data_processed.head(5)

,lat,lon,l2,l3,rooms,bedrooms,bathrooms,surface_total,surface_covered,property_type,price,days_since_start,days_since_end,available_publication
0,-58.442399,-34.573623,Capital Federal,Colegiales,3,2,2,117.95,87.4,Departamento,259000.0,2226,NaN,True
1,-58.430493,-34.606620,Capital Federal,Almagro,3,2,2,77.00,67.0,Departamento,235500.0,2037,2034.0,False
2,-58.491760,-34.574123,Capital Federal,Villa Urquiza,2,1,1,60.00,55.0,Departamento,175000.0,2011,NaN,True
3,-58.420737,-34.631770,Capital Federal,Boedo,2,1,1,74.00,47.0,PH,140000.0,2330,NaN,True
4,-58.429983,-34.607225,Capital Federal,Almagro,3,2,1,66.00,64.0,Departamento,173000.0,2342,2316.0,False


In [3]:
data_processed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 222955 entries, 0 to 992094
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   lat                    202076 non-null  float64
 1   lon                    202076 non-null  float64
 2   l2                     222955 non-null  object 
 3   l3                     222955 non-null  object 
 4   rooms                  222955 non-null  int64  
 5   bedrooms               222955 non-null  int64  
 6   bathrooms              222955 non-null  int64  
 7   surface_total          222955 non-null  float64
 8   surface_covered        222955 non-null  float64
 9   property_type          222955 non-null  object 
 10  price                  222955 non-null  float64
 11  days_since_start       222955 non-null  int64  
 12  days_since_end         159569 non-null  float64
 13  available_publication  222955 non-null  bool   
dtypes: bool(1), float64(6), int64(4), object(

In [4]:
l3_by_l2 = (
    data_processed.groupby("l2")["l3"]
    .unique()
    .apply(list)
    .to_dict()
)

l3_by_l2

{'Bs.As. G.B.A. Zona Norte': ['Tigre',
  'San Fernando',
  'Pilar',
  'San Isidro',
  'Vicente López',
  'Malvinas Argentinas',
  'General San Martín',
  'Escobar',
  'San Miguel',
  'José C Paz'],
 'Bs.As. G.B.A. Zona Oeste': ['Moreno',
  'Ituzaingó',
  'Tres de Febrero',
  'Morón',
  'La Matanza',
  'Merlo',
  'Hurlingham',
  'Marcos Paz',
  'General Rodríguez'],
 'Bs.As. G.B.A. Zona Sur': ['La Plata',
  'Lanús',
  'Ezeiza',
  'Quilmes',
  'Esteban Echeverría',
  'Lomas de Zamora',
  'Avellaneda',
  'Berazategui',
  'Almirante Brown',
  'San Vicente',
  'Cañuelas',
  'Presidente Perón',
  'Florencio Varela'],
 'Capital Federal': ['Colegiales',
  'Almagro',
  'Villa Urquiza',
  'Boedo',
  'San Telmo',
  'Barrio Norte',
  'Villa Devoto',
  'Palermo',
  'Monserrat',
  'Caballito',
  'Villa Ortuzar',
  'Villa Crespo',
  'Once',
  'Saavedra',
  'Recoleta',
  'Villa General Mitre',
  'Villa del Parque',
  'Belgrano',
  'Mataderos',
  'Balvanera',
  'Floresta',
  'Coghlan',
  'Flores',
  'V

In [5]:
with open("l3_by_l2.json", "w") as handle:
    json.dump(l3_by_l2, handle, indent=4)

In [6]:
data_processed = pd.get_dummies(data_processed, drop_first=True, dtype="int64")
data_processed.columns

Index(['lat', 'lon', 'rooms', 'bedrooms', 'bathrooms', 'surface_total',
       'surface_covered', 'price', 'days_since_start', 'days_since_end',
       ...
       'l3_Villa Real', 'l3_Villa Riachuelo', 'l3_Villa Santa Rita',
       'l3_Villa Soldati', 'l3_Villa Urquiza', 'l3_Villa del Parque',
       'property_type_Departamento', 'property_type_Local comercial',
       'property_type_Oficina', 'property_type_PH'],
      dtype='object', length=107)

Guardo los precios por cuartil, para implementar un límite en los modelos que entrene

In [7]:
price_col = data_processed["price"]

price_by_quantile = {
    "Q1": price_col.quantile(0.25),
    "Q2": price_col.quantile(0.50),
    "Q3": price_col.quantile(0.75),
    "IQR": price_col.quantile(0.75) - price_col.quantile(0.25)
}

price_by_quantile["Q3+1.5IQR"] = price_by_quantile["Q3"] + (1.5 * price_by_quantile["IQR"])
price_by_quantile

{'Q1': np.float64(96671.0),
 'Q2': np.float64(145000.0),
 'Q3': np.float64(220000.0),
 'IQR': np.float64(123329.0),
 'Q3+1.5IQR': np.float64(404993.5)}

In [8]:
with open("price_by_quantile.json", "w") as handle:
    json.dump(price_by_quantile, handle, indent=4)

Guardo los valores mínimos y máximos de cada columna

In [9]:
data_min_dict = data_processed.min().to_dict()
data_max_dict = data_processed[data_processed["price"] < price_by_quantile["Q3"]].max().to_dict()

min_max_input_values = {}

for key in data_min_dict:
    min_max_input_values[key] = {
        "Min": data_min_dict[key],
        "Max": data_max_dict[key],
    }
    
min_max_input_values

{'lat': {'Min': -59.0574481271, 'Max': -57.8305701},
 'lon': {'Min': -35.05879211, 'Max': -34.0488002},
 'rooms': {'Min': 1, 'Max': 8},
 'bedrooms': {'Min': 0, 'Max': 6},
 'bathrooms': {'Min': 1, 'Max': 4},
 'surface_total': {'Min': 0.0, 'Max': 505.0},
 'surface_covered': {'Min': 1.0, 'Max': 245.0},
 'price': {'Min': 0.0, 'Max': 219999.0},
 'days_since_start': {'Min': 1985, 'Max': 2374},
 'days_since_end': {'Min': 1925.0, 'Max': 2373.0},
 'available_publication': {'Min': False, 'Max': True},
 'l2_Bs.As. G.B.A. Zona Oeste': {'Min': 0, 'Max': 1},
 'l2_Bs.As. G.B.A. Zona Sur': {'Min': 0, 'Max': 1},
 'l2_Capital Federal': {'Min': 0, 'Max': 1},
 'l3_Agronomía': {'Min': 0, 'Max': 1},
 'l3_Almagro': {'Min': 0, 'Max': 1},
 'l3_Almirante Brown': {'Min': 0, 'Max': 1},
 'l3_Avellaneda': {'Min': 0, 'Max': 1},
 'l3_Balvanera': {'Min': 0, 'Max': 1},
 'l3_Barracas': {'Min': 0, 'Max': 1},
 'l3_Barrio Norte': {'Min': 0, 'Max': 1},
 'l3_Belgrano': {'Min': 0, 'Max': 1},
 'l3_Berazategui': {'Min': 0, 'Max

In [10]:
# Filtrar los que sean mayores a Q3
with open("min_max_input_values.json", "w") as handle:
    json.dump(min_max_input_values, handle, indent=4)

Guardo las columnas que usaré en mi modelo

In [11]:
colums_to_drop = [
    "price",
    "available_publication",
    "days_since_start",
    "days_since_end"
]

with open("categories_ohe.pkl", "wb") as handle:
    pickle.dump(data_processed.drop(columns=colums_to_drop).columns, handle, protocol=pickle.HIGHEST_PROTOCOL)

Guardo el nuevo csv

In [12]:
data_processed.to_csv("../data/03-encoded-properati.csv")